# ADRec Experiments on Amazon Baby

Run independent full-training experiments for seeds 42, 43, 44, 45, and 46. Each seed trains its own item embeddings and ADRec model. A fresh batch trains all five seeds. Existing local checkpoint folders are not modified.

Before running:

1. Import this notebook into Kaggle.
2. Attach the updated `cmpe257_kaggle_code.zip` and `dataset.pkl`.
3. Enable Internet for package installation and select a GPU.

Run the numbered sections in order. Source ZIPs and extracted source folders are both supported.


## 1. Load the project files

Locate the code and dataset under `/kaggle/input`. Extract or copy the code into `/kaggle/working/CMPE257_ML`, verify the source hashes, and add the project to the Python import path.

Leave the path variables as `None` for automatic discovery. When multiple matching inputs are attached, set `CODE_ZIP` or `CODE_DIR`, and `DATA_FILE`, to the required paths.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# None enables automatic discovery within the attached Kaggle inputs.
# Set explicit paths when multiple matching files are attached.
CODE_ZIP = None  # Path to the attached code ZIP
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATA_FILE = None  # Path to the attached Baby dataset
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

DATASET_PATH = attached_file('dataset.pkl', DATA_FILE)
REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
print('Dataset:', DATASET_PATH)


## 2. Install dependencies

Install the packages listed in `requirements-shared.txt`. The installation retains the existing CUDA-enabled PyTorch package when it satisfies the requirements. Complete this cell before importing the models.

If installation requests a kernel restart, restart the session and rerun the setup and installation cells.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


## 3. Check the runtime

Import the shared data, training, and inference functions. Check the printed PyTorch version and GPU information. Full training requires `CUDA available: True`; smoke runs also support CPU execution.


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything
from src.training.adrec_experiments import run_seed, summarize

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Check model imports before starting training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 4. Configure the seed runs

`SEEDS` lists the five experiments. A fresh execution trains seeds 42–46. Saved stages are reused only when present in the current output directory or supplied through `RESUME_ROOT`. Full mode uses all 11,761 users, hidden size 128, 32 diffusion steps, and up to 500 epochs per stage with early stopping.

Each seed is applied to both embedding pretraining and ADRec training. The dataset splits and evaluation settings remain fixed. Runs are saved separately as `runs/adrec_full_seed<N>/`.

Set `RUN_MODE = 'smoke'` for a nine-user pipeline check. Smoke outputs use separate folders and are never combined with full results. Set `RESUME_ROOT` only when attaching previously saved run folders from another session.


In [ ]:
RUN_MODE = 'full'
SEEDS = [42, 43, 44, 45, 46]
RESUME_ROOT = None  # Folder containing saved adrec_full_seed<N> run directories

assert RUN_MODE in ('full', 'smoke')
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert all(type(seed) is int and seed in (42, 43, 44, 45, 46) for seed in SEEDS)
if RUN_MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Full training requires a GPU.')
if RESUME_ROOT is not None and not Path(RESUME_ROOT).is_dir():
    raise FileNotFoundError(RESUME_ROOT)

RUNS_ROOT = WORK_ROOT / 'runs'
RESULTS_DIR = WORK_ROOT / ('results' if RUN_MODE == 'full' else 'smoke_results') / 'adrec'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg = make_config('adrec', RUN_MODE)
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print('New seeds:', SEEDS)
print('Mode:', RUN_MODE)
print('Results directory:', RESULTS_DIR)
print(dataset_summary(data))
print(json.dumps(cfg, indent=2))


## 5. Train and export seeds 42–46

For each seed, run the following stages in sequence:

1. Train the original pretraining model with cross-entropy loss and select its checkpoint using validation.
2. Load that seed's pretrained item embeddings into ADRec.
3. Keep the embeddings frozen for five full-training epochs, then train the embeddings and backbone together.
4. Select the ADRec checkpoint using validation and evaluate it on the test set.
5. Export the result files, update the CSV summaries, and create a checkpoint ZIP for the completed seed.

The loop uses the shared training and evaluation code. Each completed epoch saves `latest.pt`. Rerunning the cell reuses completed stages and resumes interrupted stages found in the same output directory. Completed seeds are not trained again. A new Kaggle session requires saved run folders attached through `RESUME_ROOT`.

The output files contain actual completed results only. Each row contains the result from its own trained model.


In [ ]:
completed_runs = {}
results_archive = None
for seed in SEEDS:
    outcome = run_seed(seed, RUN_MODE, DATASET_PATH, RUNS_ROOT, RESULTS_DIR,
                       resume_root=RESUME_ROOT)
    completed_runs[seed] = outcome
    run_dir = Path(outcome['run_dir'])
    checkpoint_archive = shutil.make_archive(str(WORK_ROOT / run_dir.name), 'zip', root_dir=run_dir)
    archive_name = 'adrec_results' if RUN_MODE == 'full' else 'adrec_smoke_results'
    results_archive = shutil.make_archive(str(WORK_ROOT / archive_name), 'zip', root_dir=RESULTS_DIR)
    print('Completed seed:', seed)
    print('Checkpoint archive:', checkpoint_archive)
    print('Results archive:', results_archive)
    print(json.dumps(outcome['result']['test'], indent=2))


## 6. Inspect the results and summary

`adrec_baby_runs.csv` contains one row per seed with the selected epoch and all six test metrics. `adrec_baby_summary.csv` uses the same columns as the baseline summaries and reports the mean and sample standard deviation (`n-1`). Values are percentages rounded to four decimal places. A single completed run has no standard-deviation row.

A complete comparison contains seeds 42–46 and `n_runs = 5`. Variation across seeds measures sensitivity to random initialization, training order, dropout, and diffusion noise. It does not measure variation across different datasets.


In [ ]:
rows = summarize(RESULTS_DIR)
print('Completed seeds:', [row['seed'] for row in rows])
print('Per-seed results:')
print((RESULTS_DIR / 'adrec_baby_runs.csv').read_text())
print('Summary:')
print((RESULTS_DIR / 'adrec_baby_summary.csv').read_text())

archive_name = 'adrec_results' if RUN_MODE == 'full' else 'adrec_smoke_results'
results_archive = shutil.make_archive(str(WORK_ROOT / archive_name), 'zip', root_dir=RESULTS_DIR)
print('Results archive:', results_archive)


## 7. Load a saved model and generate recommendations

Load the first completed seed's `best.pt` and predict ten product IDs from the visible validation history. The held-out target is excluded. Padding ID 0 is removed from this inference example; benchmark evaluation retains its original candidate rules.


In [ ]:
INFERENCE_SEED = SEEDS[0]
MODEL_DIR = RUNS_ROOT / f'adrec_{RUN_MODE}_seed{INFERENCE_SEED}' / 'adrec'
model, saved_cfg = load_for_inference(MODEL_DIR / 'best.pt')
example = SequenceDataset(data, 'val', saved_cfg['max_len'])[0]
seed_everything(saved_cfg['seed'])
scores = predict_scores(model, [example['input_ids']]).cpu()
scores[:, 0] = -torch.inf
recommended_ids = torch.topk(scores, k=10, dim=-1).indices[0].tolist()
print('Seed:', INFERENCE_SEED)
print('Visible history:', [item for item in example['input_ids'] if item])
print('Recommended product IDs:', recommended_ids)


## 8. Download the output files

Download `adrec_results.zip` and each `adrec_full_seed<N>.zip` from Kaggle's `/kaggle/working` output file browser before ending the session. The ZIP files are created after each completed seed.

Extract `adrec_results.zip` into the repository's `results/adrec/` folder. It contains:

- `adrec_seed<N>_results.json`, `adrec_seed<N>_config.json`, and `adrec_seed<N>_history.json`.
- Per-seed environment and requirements files, plus baseline-compatible `environment.json` and `requirements-lock.txt` copies from the lowest listed seed.
- `adrec_baby_runs.csv`, `adrec_baby_summary.csv`, and `EXPERIMENT.md`.
- Checkpoint SHA-256 records and the pretraining metadata under `pretrain/`.

The result archive contains no weights. Extract each checkpoint ZIP into `checkpoints/adrec_full_seed<N>/`; it contains that seed's `pretrain/` and `adrec/` folders, including `best.pt` and `latest.pt`. Keep previous experiment folders separate from the new batch.

Kaggle **Save & Run All** starts a separate execution from the first cell. Background execution uses the attached inputs and the configured resume folders; it does not inherit the current interactive session's files.


## 9. Resume interrupted experiments

1. Save the existing `runs/adrec_full_seed<N>/` folders before ending the current session, including any interrupted stage's `latest.pt` and metadata.
2. Attach the saved folders in the next Kaggle session.
3. Set `RESUME_ROOT` to their parent folder. The expected layout is `<RESUME_ROOT>/adrec_full_seed42/pretrain/` and `<RESUME_ROOT>/adrec_full_seed42/adrec/`, with the same layout for the other saved seeds.
4. Retain the original run mode and configuration. Run the notebook from the setup section.

Completed stages with matching configurations are copied to the working directory and reused. Interrupted stages resume from their latest completed epoch. Seeds without saved stages start from scratch. Early-stopped stages remain stopped. Hardware or library changes can affect numerical reproducibility.
